# Fake News Detection - Phase 2: Feature Extraction and Model Training

In this notebook, we will:
1. Load the preprocessed dataset (`processed_data.csv`).
2. Train-test split the dataset.
3. Extract features using a TF-IDF (Term Frequency-Inverse Document Frequency) Vectorizer.
4. Train a Logistic Regression Classifier.
5. Save the trained vectorizer, model, and class label encoder.

In [ ]:
import os
import sys
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

# Append src folder to path
sys.path.append(os.path.abspath('../src'))
from feature_extraction import fit_transform_tfidf, save_vectorizer

## 1. Load Preprocessed Data

In [ ]:
if not os.path.exists('../dataset/processed_data.csv'):
    print("Preprocessed data not found. Running pre-processing pipeline script...")
    # Execute preprocessing script if necessary
    import subprocess
    subprocess.run([sys.executable, '../src/data_preprocessing.py'])

df = pd.read_csv('../dataset/processed_data.csv')
df.dropna(subset=['cleaned_content'], inplace=True)
print(f"Loaded dataset shape: {df.shape}")

## 2. Train-Test Split and Label Encoding

In [ ]:
X = df['cleaned_content']
y = df['label'] # 1 is True, 0 is Fake

# Encode targets to string categories for transparency
encoder = LabelEncoder()
y_str = y.map({0: 'Fake', 1: 'True'})
y_encoded = encoder.fit_transform(y_str)

# Split into train/test partitions
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f"Training set rows: {len(X_train)}")
print(f"Testing set rows: {len(X_test)}")

## 3. TF-IDF Feature Extraction

In [ ]:
# Convert text lists into TF-IDF representations
X_train_features, vectorizer = fit_transform_tfidf(X_train)
X_test_features = vectorizer.transform(X_test)

print(f"TF-IDF Matrix feature dimensions: {X_train_features.shape[1]}")

## 4. Train and Compare Models

In [ ]:
# Candidate 1: Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_features, y_train)
y_pred_lr = lr.predict(X_test_features)
acc_lr = accuracy_score(y_test, y_pred_lr)

# Candidate 2: Multinomial Naive Bayes
nb = MultinomialNB()
nb.fit(X_train_features, y_train)
y_pred_nb = nb.predict(X_test_features)
acc_nb = accuracy_score(y_test, y_pred_nb)

print(f"Logistic Regression Test Accuracy: {acc_lr * 100:.2f}%")
print(f"Multinomial Naive Bayes Test Accuracy: {acc_nb * 100:.2f}%")

## 5. Exporting trained model and assets

In [ ]:
# Save chosen Logistic Regression model and vectorizer
os.makedirs('../models', exist_ok=True)
save_vectorizer(vectorizer, filepath='../models/vectorizer.pkl')
joblib.dump(lr, '../models/fake_news_model.pkl')
joblib.dump(encoder, '../models/label_encoder.pkl')

# Also save the split test data so we can evaluate it separately in next notebook
test_df = pd.DataFrame({
    'cleaned_content': X_test,
    'label': y_test
})
test_df.to_csv('../dataset/test_data.csv', index=False)

print("Artifacts exported to models/ and dataset/test_data.csv successfully!")